# Evaluate the fine-tuned Qwen2.5-7B judges — DataSphere (one-pass)

Loads the fine-tuned models from the Hugging Face Hub and scores them on the
held-out test split (same `split_samples(seed=42)` used in training), with the
repo's own `parse_prediction` + `metrics.evaluate_predictions`.

**No paid storage needed** — the ~15GB models cache on ephemeral `/tmp`.
Inference fits any GPU with ≥~18GB (A100 / L4 40GB comfortably). Edit the config
in the first cell, then **Run All**.

In [ ]:
# ======================= EDIT, THEN RUN ALL =======================
MODELS = {
    "direct": "andy-takker/qwen2.5-7b-rag-judge-direct",
    "marker": "andy-takker/qwen2.5-7b-rag-judge-marker",
}
HF_TOKEN = ""     # write/read token if the Hub repos are private (huggingface.co/settings/tokens)
SEED     = 42     # must match the training split to evaluate on the same held-out test set
# ==================================================================

import os, sys, subprocess
WORK = "/tmp/eval"                    # ephemeral working dir (no paid storage)
os.environ["HF_HOME"] = "/tmp/hf"     # model cache on the big ephemeral disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(WORK, exist_ok=True)

from importlib.metadata import version, PackageNotFoundError
def _v(p):
    try: return version(p)
    except PackageNotFoundError: return None
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# Same DataSphere-tested stack as training (inference-only: no trl/deepspeed/bitsandbytes).
# numpy 1.26 sentinel: the base image ships numpy 1.22, so this triggers a full install.
if (not (_v("numpy") or "").startswith("1.26")) or (_v("torch") != "2.5.1+cu121"):
    # torch under DataSphere's CUDA 12.2 driver, with matching vision/audio so the base
    # image's torch-2.0 torchaudio can't ABI-clash when transformers imports it.
    _pip("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
         "--index-url", "https://download.pytorch.org/whl/cu121")
    # accelerate is REQUIRED for from_pretrained(device_map="auto").
    _pip("transformers>=4.56.2", "accelerate>=1.4.0", "pydantic>=2.5",
         "scikit-learn", "sentencepiece", "huggingface-hub")
    _pip("numpy==1.26.4")   # LAST: keeps base soxr/scipy/sklearn ABI-compatible (<2)
    print("Installed. On a fresh kernel just continue; if torch was already imported, Restart Kernel + Run All.")
else:
    print("stack already present")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("models:", list(MODELS.values()))


In [ ]:
# --- Repo (parser + metrics + prompts) and the test split ---
REPO_URL, REPO_BRANCH = "https://github.com/aldem2k00/rag-reliability.git", "main"
REPO_DIR = f"{WORK}/rag-reliability"
import os, sys, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR])
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)

from rag_reliability.dataset import load_jsonl, split_samples
from rag_reliability.parsing import parse_prediction
from rag_reliability import metrics as M
from rag_reliability.prompts import build_direct_prompt, build_marker_prompt

# Rebuild organizers.jsonl from the committed archive, then take the same held-out test split.
DATA = f"{WORK}/organizers.jsonl"
if not os.path.exists(DATA):
    env = dict(os.environ, PYTHONPATH=src)
    subprocess.check_call([sys.executable, os.path.join(REPO_DIR, "scripts", "prepare_data.py"),
                           "--input", os.path.join(REPO_DIR, "from_organizators", "data", "data.zip"),
                           "--output", DATA], env=env)
samples = load_jsonl(DATA)
_, _, test_samples = split_samples(samples, seed=SEED)
print(f"test split: {len(test_samples)} samples (of {len(samples)})")


In [ ]:
# --- Evaluate each model on the held-out test set ---
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def eval_model(mode, model_id):
    build_prompt = build_direct_prompt if mode == "direct" else build_marker_prompt
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16,
                                                 device_map="auto", token=HF_TOKEN or None)
    model.eval()

    @torch.no_grad()
    def generate(prompt):
        enc = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                      add_generation_prompt=True, return_tensors="pt",
                                      return_dict=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=64, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
        return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    preds = [parse_prediction(generate(build_prompt(s)), s.id, expect_marker=(mode == "marker"))
             for s in test_samples]
    res = M.evaluate_predictions(test_samples, preds)
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return (res.model_dump() if hasattr(res, "model_dump") else dict(res)), preds

results, PREDS = {}, {}
for mode, model_id in MODELS.items():
    print(f"\n=== {mode}: {model_id} ===")
    results[mode], PREDS[mode] = eval_model(mode, model_id)
    print(results[mode])


In [ ]:
# --- Side-by-side summary ---
keys = ["reliable_f1_macro", "faithfulness_f1_macro", "relevance_f1_macro",
        "invalid_output_rate", "marker_f1_macro"]
w = max(len(k) for k in keys)
hdr = "metric".ljust(w) + "".join(f"{m:>14}" for m in results)
print(hdr); print("-" * len(hdr))
for k in keys:
    row = k.ljust(w)
    for m in results:
        v = results[m].get(k)
        row += f"{'—' if v is None else format(v, '.4f'):>14}"
    print(row)


In [ ]:
# --- Diagnostics: predicted-class distribution + 2x2 confusion per field ---
# f1_macro ~0.5 often means the model collapses to one class. This shows it.
from collections import Counter

def field_report(gold, pred, name):
    g, p = list(gold), list(pred)
    tp = sum(1 for a, b in zip(g, p) if a == 1 and b == 1)
    fp = sum(1 for a, b in zip(g, p) if a == 0 and b == 1)
    fn = sum(1 for a, b in zip(g, p) if a == 1 and b == 0)
    tn = sum(1 for a, b in zip(g, p) if a == 0 and b == 0)
    print(f"  {name:13s} gold={dict(sorted(Counter(g).items()))} "
          f"pred={dict(sorted(Counter(p).items()))} | TP={tp} FP={fp} FN={fn} TN={tn}")

for mode in results:
    preds = PREDS[mode]
    print(f"\n=== {mode} ===")
    field_report([s.reliable for s in test_samples],     [p.reliable_pred for p in preds],     "reliable")
    field_report([s.faithfulness for s in test_samples], [p.faithfulness_pred for p in preds], "faithfulness")
    field_report([s.relevance for s in test_samples],    [p.relevance_pred for p in preds],    "relevance")
    if mode == "marker":
        mk = Counter(p.marker_pred for p in preds)
        print("  marker preds:", dict(mk.most_common(10)))
